# 🌾 Automated Multimodal Crop Disease Diagnosis
## Wheat Disease Classification: Health / Rust / Other
### Modalities: RGB · Multispectral (5-band) · Hyperspectral (125-band)


In [ ]:
# Install required packages (run once)
import subprocess, sys
pkgs = ["rasterio", "scikit-learn", "timm", "einops", "tqdm", "matplotlib",
        "seaborn", "pandas", "numpy", "Pillow", "torch", "torchvision"]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
print("All packages ready ✓")


In [ ]:
import os, re, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import rasterio
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


In [ ]:
# ── Dataset Paths ────────────────────────────────────────────────────────────
BASE   = Path("/home/jyoti/Documents/COMPITION/Kaggle/GreenAid/data")
TRAIN  = BASE / "train"
VAL    = BASE / "val"

# Modality sub-directories
TRAIN_RGB = TRAIN / "RGB"
TRAIN_MS  = TRAIN / "MS"
TRAIN_HS  = TRAIN / "HS"
VAL_RGB   = VAL / "RGB"
VAL_MS    = VAL / "MS"
VAL_HS    = VAL / "HS"

CLASSES    = ["Health", "Rust", "Other"]
CLS2IDX    = {c: i for i, c in enumerate(CLASSES)}

# Hyper-parameters
IMG_SIZE   = 64        # spatial resize for all modalities
BATCH_SIZE = 32
LR         = 3e-4
EPOCHS     = 30
HS_BANDS   = 125
HS_USEFUL  = slice(10, 111)   # drop noisy ends → 101 bands
MS_BANDS   = 5

print("Paths set ✓")
print("Train RGB:", len(list(TRAIN_RGB.glob("*.png"))) if TRAIN_RGB.exists() else "N/A")
print("Train MS :", len(list(TRAIN_MS.glob("*.tif"))))
print("Train HS :", len(list(TRAIN_HS.glob("*.tif"))))
print("Val   MS :", len(list(VAL_MS.glob("*.tif"))))


In [ ]:
# ── Exploratory Data Analysis ────────────────────────────────────────────────
def count_by_class(directory, ext):
    counts = {}
    for cls in CLASSES:
        pattern = f"{cls}_*.{ext}" if "tif" in ext else f"{cls}_*.{ext}"
        counts[cls] = len(list(Path(directory).glob(pattern)))
    return counts

ms_counts  = count_by_class(TRAIN_MS, "tif")
hs_counts  = count_by_class(TRAIN_HS, "tif")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
colors = ["#4CAF50", "#F44336", "#FF9800"]

ax[0].bar(CLASSES, [ms_counts[c] for c in CLASSES], color=colors)
ax[0].set_title("MS Training Samples per Class"); ax[0].set_ylabel("Count")
ax[1].bar(CLASSES, [hs_counts[c] for c in CLASSES], color=colors)
ax[1].set_title("HS Training Samples per Class")
for a in ax:
    for bar, v in zip(a.patches, [ms_counts[c] for c in CLASSES]):
        a.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
               str(bar.get_height()), ha="center", fontsize=11)
plt.tight_layout(); plt.savefig("class_distribution.png", dpi=100); plt.show()
print("Class distribution:", ms_counts)


In [ ]:
# ── Visualise Sample Images ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
mod_names = ["MS (False-Color RGB)", "HS (Band-50 mid-vis)", "RGB"]

for ci, cls in enumerate(CLASSES):
    # MS  → use bands 2,1,0 (R,G,B) of the 5-band file
    ms_file = sorted(TRAIN_MS.glob(f"{cls}_*.tif"))[0]
    with rasterio.open(ms_file) as src:
        ms = src.read().astype(np.float32)   # (5, H, W)
    rgb_false = np.stack([ms[2], ms[1], ms[0]], axis=-1)
    rgb_false = (rgb_false - rgb_false.min()) / (rgb_false.max() - rgb_false.min() + 1e-6)
    axes[ci][0].imshow(rgb_false); axes[ci][0].set_title(f"{cls} – MS"); axes[ci][0].axis("off")

    # HS → band 50
    hs_file = sorted(TRAIN_HS.glob(f"{cls}_*.tif"))[0]
    with rasterio.open(hs_file) as src:
        hs = src.read().astype(np.float32)
    band = hs[50]; band = (band - band.min()) / (band.max() - band.min() + 1e-6)
    axes[ci][1].imshow(band, cmap="RdYlGn"); axes[ci][1].set_title(f"{cls} – HS band50"); axes[ci][1].axis("off")

    # RGB
    rgb_files = sorted(TRAIN_RGB.glob(f"{cls}_*.png")) if TRAIN_RGB.exists() else []
    if rgb_files:
        img = np.array(Image.open(rgb_files[0]).convert("RGB"))
        axes[ci][2].imshow(img)
    else:
        axes[ci][2].text(0.4, 0.5, "N/A", transform=axes[ci][2].transAxes)
    axes[ci][2].set_title(f"{cls} – RGB"); axes[ci][2].axis("off")

plt.suptitle("Sample patches per class and modality", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.savefig("sample_vis.png", dpi=100); plt.show()


In [ ]:
# ── Spectral Signatures ──────────────────────────────────────────────────────
wavelengths = np.linspace(450, 950, 125)  # ~4nm spacing

fig, ax = plt.subplots(figsize=(12, 5))
for cls, color in zip(CLASSES, ["green", "red", "orange"]):
    spectra = []
    for f in list(TRAIN_HS.glob(f"{cls}_*.tif"))[:20]:
        with rasterio.open(f) as src:
            data = src.read().astype(np.float32)   # (125, H, W)
        spectra.append(data.reshape(125, -1).mean(axis=1))
    mean_s = np.mean(spectra, axis=0)
    std_s  = np.std(spectra,  axis=0)
    ax.plot(wavelengths, mean_s, label=cls, color=color, linewidth=2)
    ax.fill_between(wavelengths, mean_s - std_s, mean_s + std_s,
                    alpha=0.2, color=color)

ax.axvspan(450, 510, alpha=0.05, color="blue", label="Blue")
ax.axvspan(530, 570, alpha=0.05, color="green")
ax.axvspan(630, 680, alpha=0.05, color="red")
ax.axvspan(700, 760, alpha=0.05, color="purple", label="Red-Edge")
ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Mean Reflectance")
ax.set_title("Mean Spectral Signature per Class (HS)")
ax.legend(); plt.tight_layout()
plt.savefig("spectral_signatures.png", dpi=100); plt.show()


In [ ]:
# ── Multimodal Dataset ───────────────────────────────────────────────────────
class WheatDataset(Dataset):
    """
    Loads aligned RGB + MS + HS tiles for train split.
    Naming: {Class}_hyper_{N}.tif  /  {Class}_multi_{N}.tif  /  {Class}_{N}.png
    Falls back gracefully if RGB dir absent.
    """
    def __init__(self, ms_dir, hs_dir, rgb_dir=None, img_size=64,
                 augment=False, hs_bands_slice=slice(10, 111)):
        self.ms_dir   = Path(ms_dir)
        self.hs_dir   = Path(hs_dir)
        self.rgb_dir  = Path(rgb_dir) if rgb_dir else None
        self.img_size = img_size
        self.augment  = augment
        self.hs_slice = hs_bands_slice

        self.samples = []   # (ms_path, hs_path, rgb_path_or_None, label)
        for cls in CLASSES:
            ms_files = sorted(self.ms_dir.glob(f"{cls}_*.tif"))
            for ms_f in ms_files:
                # derive HS companion
                num = re.search(r"_(\d+)\.tif$", ms_f.name)
                if not num: continue
                n = num.group(1)
                hs_f = self.hs_dir / f"{cls}_hyper_{n}.tif"
                if not hs_f.exists(): continue
                # RGB companion (optional)
                rgb_f = None
                if self.rgb_dir and self.rgb_dir.exists():
                    cands = list(self.rgb_dir.glob(f"{cls}_*{n}.png"))
                    if cands: rgb_f = cands[0]
                self.samples.append((ms_f, hs_f, rgb_f, CLS2IDX[cls]))

        self.tf_spatial = T.Compose([
            T.Resize((img_size, img_size)),
            T.RandomHorizontalFlip() if augment else T.Lambda(lambda x: x),
            T.RandomVerticalFlip()   if augment else T.Lambda(lambda x: x),
        ])

    def __len__(self): return len(self.samples)

    def _read_tif(self, path, bands_slice=None):
        with rasterio.open(path) as src:
            data = src.read().astype(np.float32)   # (C, H, W)
        if bands_slice: data = data[bands_slice]
        # per-channel min-max normalise
        mn = data.min(axis=(1,2), keepdims=True)
        mx = data.max(axis=(1,2), keepdims=True)
        data = (data - mn) / (mx - mn + 1e-6)
        return data

    def __getitem__(self, idx):
        ms_f, hs_f, rgb_f, label = self.samples[idx]

        ms = self._read_tif(ms_f)                         # (5, H, W)
        hs = self._read_tif(hs_f, self.hs_slice)          # (101, H, W)

        # Resize via PIL torchvision path
        ms_t = torch.from_numpy(ms)  # keep channels, resize spatially
        hs_t = torch.from_numpy(hs)

        def resize_mc(tensor):
            """Resize a (C,H,W) tensor to img_size × img_size."""
            return F.interpolate(tensor.unsqueeze(0), size=(self.img_size, self.img_size),
                                 mode="bilinear", align_corners=False).squeeze(0)

        ms_t = resize_mc(ms_t)
        hs_t = resize_mc(hs_t)

        # RGB
        if rgb_f:
            img = np.array(Image.open(rgb_f).convert("RGB")).astype(np.float32) / 255.
            rgb_t = torch.from_numpy(img.transpose(2,0,1))   # (3, H, W)
            rgb_t = resize_mc(rgb_t)
        else:
            rgb_t = ms_t[:3].clone()  # synthetic RGB from first 3 MS bands

        # Augment: random 90° rotation
        if self.augment and random.random() > 0.5:
            k = random.randint(1, 3)
            ms_t  = torch.rot90(ms_t,  k, [1,2])
            hs_t  = torch.rot90(hs_t,  k, [1,2])
            rgb_t = torch.rot90(rgb_t, k, [1,2])

        return {"ms": ms_t, "hs": hs_t, "rgb": rgb_t, "label": torch.tensor(label)}


In [ ]:
# ── Validation / Test Dataset (no labels) ───────────────────────────────────
class WheatValDataset(Dataset):
    """Unlabelled validation set – for generating submission CSV."""
    def __init__(self, ms_dir, hs_dir, rgb_dir=None, img_size=64,
                 hs_bands_slice=slice(10, 111)):
        self.ms_dir  = Path(ms_dir)
        self.hs_dir  = Path(hs_dir)
        self.rgb_dir = Path(rgb_dir) if rgb_dir else None
        self.img_size = img_size
        self.hs_slice = hs_bands_slice

        self.ids = sorted([f.stem for f in self.ms_dir.glob("val_*.tif")])

    def __len__(self): return len(self.ids)

    def _read_tif(self, path, bands_slice=None):
        with rasterio.open(path) as src:
            data = src.read().astype(np.float32)
        if bands_slice: data = data[bands_slice]
        mn = data.min(axis=(1,2), keepdims=True)
        mx = data.max(axis=(1,2), keepdims=True)
        return (data - mn) / (mx - mn + 1e-6)

    def __getitem__(self, idx):
        vid = self.ids[idx]
        ms_f = self.ms_dir / f"{vid}.tif"
        hs_f = self.hs_dir / f"{vid}.tif"

        ms = torch.from_numpy(self._read_tif(ms_f))
        hs = torch.from_numpy(self._read_tif(hs_f, self.hs_slice))

        def resize_mc(t):
            return F.interpolate(t.unsqueeze(0), size=(self.img_size, self.img_size),
                                 mode="bilinear", align_corners=False).squeeze(0)
        ms = resize_mc(ms); hs = resize_mc(hs)

        if self.rgb_dir and (self.rgb_dir / f"{vid}.png").exists():
            img = np.array(Image.open(self.rgb_dir / f"{vid}.png").convert("RGB")).astype(np.float32)/255.
            rgb = resize_mc(torch.from_numpy(img.transpose(2,0,1)))
        else:
            rgb = ms[:3].clone()

        return {"ms": ms, "hs": hs, "rgb": rgb, "id": vid}


In [ ]:
# ── Multimodal Fusion Model ──────────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction), nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x).view(-1, x.shape[1], 1, 1)


class ModalityEncoder(nn.Module):
    """Lightweight CNN encoder for one modality."""
    def __init__(self, in_ch, out_ch=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 64, 3, padding=1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64,  64, 3, padding=1, groups=4), nn.BatchNorm2d(64), nn.GELU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.GELU(),
            ChannelAttention(out_ch),
            nn.AdaptiveAvgPool2d(4),   # → (out_ch, 4, 4)
        )
    def forward(self, x): return self.net(x)


class MultimodalCropNet(nn.Module):
    """
    Fuses RGB (3ch) + MS (5ch) + HS (101ch) via separate encoders
    + cross-modal attention + MLP classifier.
    """
    def __init__(self, n_classes=3, feat_dim=128, hs_ch=101, ms_ch=5):
        super().__init__()
        self.rgb_enc = ModalityEncoder(3,     feat_dim)
        self.ms_enc  = ModalityEncoder(ms_ch, feat_dim)
        self.hs_enc  = ModalityEncoder(hs_ch, feat_dim)

        flat = feat_dim * 4 * 4   # 128 * 16 = 2048

        # Cross-modal attention (query=fused, key/val=hs)
        self.attn = nn.MultiheadAttention(embed_dim=flat, num_heads=8, batch_first=True)

        self.fusion = nn.Sequential(
            nn.Linear(flat * 3, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )

    def forward(self, rgb, ms, hs):
        f_rgb = self.rgb_enc(rgb).flatten(1)   # (B, flat)
        f_ms  = self.ms_enc(ms).flatten(1)
        f_hs  = self.hs_enc(hs).flatten(1)

        # Stack for attention: treat modalities as a sequence of length 3
        seq = torch.stack([f_rgb, f_ms, f_hs], dim=1)   # (B, 3, flat)
        attn_out, _ = self.attn(seq, seq, seq)            # (B, 3, flat)
        fused = attn_out.reshape(attn_out.size(0), -1)    # (B, 3*flat)

        return self.fusion(fused)

model = MultimodalCropNet(n_classes=3, feat_dim=128, hs_ch=101, ms_ch=MS_BANDS).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: MultimodalCropNet | Trainable params: {total_params:,}")


In [ ]:
# ── Build DataLoaders ────────────────────────────────────────────────────────
full_ds = WheatDataset(TRAIN_MS, TRAIN_HS, TRAIN_RGB, img_size=IMG_SIZE,
                       augment=False, hs_bands_slice=HS_USEFUL)
print(f"Total labelled samples: {len(full_ds)}")

# 80/20 split within training data for validation
n_val  = int(0.2 * len(full_ds))
n_tr   = len(full_ds) - n_val
tr_ds, val_ds = random_split(full_ds, [n_tr, n_val],
                              generator=torch.Generator().manual_seed(SEED))

# Re-enable augmentation on train subset view
aug_ds = WheatDataset(TRAIN_MS, TRAIN_HS, TRAIN_RGB, img_size=IMG_SIZE,
                      augment=True, hs_bands_slice=HS_USEFUL)
aug_ds.samples = [full_ds.dataset.samples[i] for i in tr_ds.indices]
aug_ds.tf_spatial = full_ds.tf_spatial   # keep img_size

tr_loader  = DataLoader(aug_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(aug_ds)}  |  Val: {len(val_ds)}")
print(f"Batches – train: {len(tr_loader)}  val: {len(val_loader)}")


In [ ]:
# ── Training Utilities ──────────────────────────────────────────────────────
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# Class weights for imbalance safety (balanced here, but good practice)
cls_counts = np.array([len(list(TRAIN_MS.glob(f"{c}_*.tif"))) for c in CLASSES], dtype=float)
class_weights = torch.tensor(cls_counts.sum() / (len(CLASSES) * cls_counts), dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1, eta_min=1e-6)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tot_loss, correct, total = 0., 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            rgb   = batch["rgb"].to(DEVICE)
            ms    = batch["ms"].to(DEVICE)
            hs    = batch["hs"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            if train: optimizer.zero_grad()
            logits = model(rgb, ms, hs)
            loss   = criterion(logits, labels)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            tot_loss += loss.item() * labels.size(0)
            preds     = logits.argmax(dim=1)
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)
    return tot_loss / total, correct / total

print("Training utilities ready ✓")


In [ ]:
# ── Training Loop ────────────────────────────────────────────────────────────
history = {"tr_loss": [], "tr_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.
SAVE_PATH = "best_multimodal_model.pt"

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc   = run_epoch(tr_loader,  train=True)
    val_loss, val_acc  = run_epoch(val_loader, train=False)
    scheduler.step(epoch)

    history["tr_loss"].append(tr_loss);   history["tr_acc"].append(tr_acc)
    history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        mark = " ★ saved"
    else:
        mark = ""

    if epoch % 5 == 0 or epoch == 1:
        print(f"Ep {epoch:3d}/{EPOCHS} | "
              f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}{mark}")

print(f"\nBest val accuracy: {best_val_acc:.4f}")


In [ ]:
# ── Training Curves ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, EPOCHS + 1)

axes[0].plot(ep, history["tr_loss"], label="Train"); axes[0].plot(ep, history["val_loss"], label="Val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(ep, history["tr_acc"], label="Train"); axes[1].plot(ep, history["val_acc"], label="Val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()

plt.suptitle("Training History – MultimodalCropNet", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig("training_curves.png", dpi=100); plt.show()


In [ ]:
# ── Detailed Evaluation (internal val split) ────────────────────────────────
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        logits = model(batch["rgb"].to(DEVICE), batch["ms"].to(DEVICE), batch["hs"].to(DEVICE))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(batch["label"].numpy())

print(classification_report(all_labels, all_preds, target_names=CLASSES))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix (val acc={accuracy_score(all_labels,all_preds):.3f})")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=100); plt.show()


In [ ]:
# ── Generate Submission CSV ──────────────────────────────────────────────────
test_ds     = WheatValDataset(VAL_MS, VAL_HS, VAL_RGB, img_size=IMG_SIZE, hs_bands_slice=HS_USEFUL)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

IDX2CLS = {v: k for k, v in CLS2IDX.items()}
model.eval()

rows = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        logits = model(batch["rgb"].to(DEVICE), batch["ms"].to(DEVICE), batch["hs"].to(DEVICE))
        preds  = logits.argmax(1).cpu().numpy()
        for vid, pred in zip(batch["id"], preds):
            rows.append({"Id": vid + ".tif", "Category": IDX2CLS[pred]})

sub = pd.DataFrame(rows).sort_values("Id").reset_index(drop=True)
sub.to_csv("submission.csv", index=False)
print(f"Saved submission.csv  ({len(sub)} rows)")
print(sub["Category"].value_counts())
sub.head(10)


In [ ]:
# ── Optional: Test-Time Augmentation (TTA) ──────────────────────────────────
# Improves robustness by averaging predictions over 4 flipped versions

def predict_tta(batch):
    """Average logits over 4 augmentations (original + 3 rotations)."""
    logits_sum = 0
    for k in range(4):
        rgb_k = torch.rot90(batch["rgb"], k, [2,3]).to(DEVICE)
        ms_k  = torch.rot90(batch["ms"],  k, [2,3]).to(DEVICE)
        hs_k  = torch.rot90(batch["hs"],  k, [2,3]).to(DEVICE)
        logits_sum += model(rgb_k, ms_k, hs_k).softmax(-1)
    return (logits_sum / 4).argmax(-1)

model.eval()
rows_tta = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="TTA Predicting"):
        preds = predict_tta(batch).cpu().numpy()
        for vid, pred in zip(batch["id"], preds):
            rows_tta.append({"Id": vid + ".tif", "Category": IDX2CLS[pred]})

sub_tta = pd.DataFrame(rows_tta).sort_values("Id").reset_index(drop=True)
sub_tta.to_csv("submission_tta.csv", index=False)
print(f"TTA submission saved ({len(sub_tta)} rows)")
print(sub_tta["Category"].value_counts())
sub_tta.head(10)


In [ ]:
# ── Bonus: Vegetation Indices from MS Bands ──────────────────────────────────
# Band order: B(0-Blue), G(1-Green), R(2-Red), RE(3-RedEdge), NIR(4-NIR)
# NDVI  = (NIR - R) / (NIR + R)
# NDRE  = (NIR - RE) / (NIR + RE)
# GNDVI = (NIR - G) / (NIR + G)

def compute_vi(ms_arr):
    """ms_arr: (5, H, W) float32, returns dict of (H,W) index arrays."""
    B, G, R, RE, NIR = [ms_arr[i] for i in range(5)]
    eps = 1e-6
    return {
        "NDVI":  (NIR - R)  / (NIR + R  + eps),
        "NDRE":  (NIR - RE) / (NIR + RE + eps),
        "GNDVI": (NIR - G)  / (NIR + G  + eps),
    }

fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ci, cls in enumerate(CLASSES):
    f = sorted(TRAIN_MS.glob(f"{cls}_*.tif"))[0]
    with rasterio.open(f) as src: ms = src.read().astype(np.float32)
    vis = compute_vi(ms)
    for ji, (vname, vdata) in enumerate(vis.items()):
        im = axes[ci][ji].imshow(vdata, cmap="RdYlGn", vmin=-1, vmax=1)
        axes[ci][ji].set_title(f"{cls} – {vname}"); axes[ci][ji].axis("off")
        plt.colorbar(im, ax=axes[ci][ji], fraction=0.046, pad=0.04)

plt.suptitle("Vegetation Indices by Class", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig("vegetation_indices.png", dpi=100); plt.show()
